In [1]:
# import Pkg; Pkg.add(["NCDatasets", "Interpolations"])
#
# Land Mask file used from:
# Mikelsons, Karlis; Wang, Menghua; Jiang, Lide; Wang, Xiao-Long (2021), 
# “Global land mask for satellite ocean color remote sensing”, 
# Mendeley Data, V1, doi: 10.17632/9r93m9s7cw.1

using NCDatasets

"""
    apply_noaa_watermask!(mask, xi, yi; path="watermask.nc", verbose=true)

Specialized for NOAA STAR watermask:
  - variables: watermask (Int8, 1=water, 0=land/ice), lon (1-D), lat (1-D)
  - dimensions order: (lon, lat)

Reads only the needed lon/lat window, snaps (xi, yi) to nearest grid cell
with direct index math, and AND-merges into `mask` in place.

Assumptions:
  - `xi`, `yi` are 1-D axes from DIVAnd_rectdom (lon-like, lat-like).
  - `mask` is Bool with size (length(xi), length(yi)).
"""
function apply_noaa_watermask!(mask::AbstractMatrix{Bool},
                               xi::AbstractVector{<:Real},
                               yi::AbstractVector{<:Real};
                               path::AbstractString="watermask.nc",
                               verbose::Bool=true)

    (size(mask,1), size(mask,2)) == (length(xi), length(yi)) ||
        error("mask must be (length(xi), length(yi)); got mask=$(size(mask)) axes=($(length(xi)),$(length(yi))).")

    xi_min, xi_max = extrema(xi)
    yi_min, yi_max = extrema(yi)

    ds = NCDataset(path, "r")
    try
        lon = vec(Array(ds["lon"]))   # expected regular: step ≈ 1/480°
        lat = vec(Array(ds["lat"]))

        # Handle descending axes in file
        lon_rev = length(lon) > 1 && lon[2] < lon[1]
        lat_rev = length(lat) > 1 && lat[2] < lat[1]
        lon_sorted = lon_rev ? reverse(lon) : lon
        lat_sorted = lat_rev ? reverse(lat) : lat

        # Step sizes (assumed constant)
        dlon = length(lon_sorted) > 1 ? lon_sorted[2] - lon_sorted[1] : 1.0
        dlat = length(lat_sorted) > 1 ? lat_sorted[2] - lat_sorted[1] : 1.0

        # Pad by one cell
        i1s = clamp(searchsortedfirst(lon_sorted, xi_min - dlon), 1, length(lon_sorted))
        i2s = clamp(searchsortedlast( lon_sorted, xi_max + dlon),  1, length(lon_sorted))
        j1s = clamp(searchsortedfirst(lat_sorted, yi_min - dlat), 1, length(lat_sorted))
        j2s = clamp(searchsortedlast( lat_sorted, yi_max + dlat),  1, length(lat_sorted))

        # Map to file indices if reversed
        if lon_rev
            Nlon = length(lon_sorted)
            i1, i2 = Nlon - i2s + 1, Nlon - i1s + 1
        else
            i1, i2 = i1s, i2s
        end
        if lat_rev
            Nlat = length(lat_sorted)
            j1, j2 = Nlat - j2s + 1, Nlat - j1s + 1
        else
            j1, j2 = j1s, j2s
        end

        verbose && println("→ Reading subset: lon[$i1:$i2] × lat[$j1:$j2]")

        wm_sub = Array(ds["watermask"][i1:i2, j1:j2])  # Int8, no casting
        lon_sub_file = lon[i1:i2]
        lat_sub_file = lat[j1:j2]

        # Re-orient subset to ascending axes for simple index math
        if lon_rev
            wm_sub = reverse(wm_sub, dims=1)
            lon_sub = reverse(lon_sub_file)
        else
            lon_sub = lon_sub_file
        end
        if lat_rev
            wm_sub = reverse(wm_sub, dims=2)
            lat_sub = reverse(lat_sub_file)
        else
            lat_sub = lat_sub_file
        end

        # Nearest-neighbor indices on regular grids (no large temp arrays)
        # idx = round( (x - x0)/dx ) + 1, clamped to [1, N]
        @inline nearest_idx(x, x0, dx, n) = clamp(round(Int, (x - x0)/dx) + 1, 1, n)

        nx, ny = length(xi), length(yi)
        ix = Vector{Int}(undef, nx)
        @inbounds for k in 1:nx
            ix[k] = nearest_idx(xi[k], lon_sub[1], dlon, length(lon_sub))
        end
        iy = Vector{Int}(undef, ny)
        @inbounds for k in 1:ny
            iy[k] = nearest_idx(yi[k], lat_sub[1], dlat, length(lat_sub))
        end

        # Gather the subset mask on your grid: this allocates an Int8 matrix of size(nx, ny)
        sel = wm_sub[ix, iy]          # Int8 values 0/1
        mask_from_file = sel .>= 1    # Bool: true where water

        # AND-merge: keep only cells that are already valid AND water
        mask .= mask .& mask_from_file
        verbose && println("✓ Watermask applied. Kept $(count(mask)) of $(length(mask)) cells.")
        return mask
    finally
        close(ds)
    end
end



apply_noaa_watermask!

In [2]:
using DIVAnd
using Rasters
using GDAL
import ArchGDAL
using Statistics

"""
    fill_missing_values(input_fname::String; lat_lon_len::Int=10, epsilon2::Float64=10.0, watermask_path::String="watermask.nc")

Reads the first band of a multi-band GeoTIFF, interpolates missing values using DIVAnd,
and saves the filled raster. Returns the output path.

# Arguments
- `input_fname::String`: Filename (without path) of the raster in `data/raw/`.
- `lat_lon_len::Int=10`: Correlation length (in pixels).
- `epsilon2::Float64=10.0`: Normalized variance of the observation error.
- `watermask_path::String="watermask.nc"`: Path to water mask NetCDF.

# Returns
- `output_path::String`: The saved interpolated raster file path.
"""
function fill_missing_values(input_fname::String; lat_lon_len::Int=10, epsilon2::Float64=10.0, watermask_path::String="watermask.nc")

    input_geotiff_path = "data/raw/$(input_fname)"
    output_geotiff_path = "data/filled/$(input_fname).filled.2D.len$(lat_lon_len).e$(epsilon2).tif"

    if !isfile(input_geotiff_path)
        error("Input file not found at: $(input_geotiff_path)")
    end

    @info "Loading GeoTIFF from: $(input_geotiff_path)"
    full_raster = Raster(input_geotiff_path)
    surface_raster = full_raster[Band(1)]

    if ndims(surface_raster) != 2
        error("Input raster is not 2D, got $(ndims(surface_raster)) dimensions.")
    end

    valid_indices = findall(!ismissing, surface_raster)
    @info "Found $(length(valid_indices)) valid points."

    nobs = length(valid_indices)
    x_obs = Vector{Float64}(undef, nobs)
    y_obs = Vector{Float64}(undef, nobs)
    f_obs = Vector{Float64}(undef, nobs)

    x_coords = dims(surface_raster, X)
    y_coords = dims(surface_raster, Y)

    for (i, idx) in enumerate(valid_indices)
        ix, iy = Tuple(idx)
        x_obs[i] = x_coords[ix]
        y_obs[i] = y_coords[iy]
        f_obs[i] = surface_raster[idx]
    end

    mean_f_obs = mean(f_obs)
    f_anomalies = f_obs .- mean_f_obs

    original_x_coords = Float64.(val(dims(surface_raster, X)))
    original_y_coords = Float64.(val(dims(surface_raster, Y)))
    grid_x = sort(original_x_coords)
    grid_y = sort(original_y_coords)

    mask, (pm, pn), (xi, yi) = DIVAnd_rectdom(grid_x, grid_y)

    xi_1d = vec(xi[:, 1])
    yi_1d = vec(yi[1, :])
    apply_noaa_watermask!(mask, xi_1d, yi_1d; path=watermask_path)

    len = (lat_lon_len, lat_lon_len)
    @time fi_anomalies, s = DIVAndrun(
        mask, (pm, pn), (xi, yi), (x_obs, y_obs), f_anomalies, len, epsilon2;
        inversion = :solve, tol = 1e-5, maxit = 2000
    )

    fi_interpolated = fi_anomalies .+ mean_f_obs

    fi_reoriented = fi_interpolated
    if original_y_coords[1] > original_y_coords[end]
        fi_reoriented = reverse(fi_interpolated, dims=2)
    end

    final_data = fi_reoriented
    input_crs = Rasters.crs(surface_raster)

    new_dims = (
        X(LinRange(original_x_coords[1], original_x_coords[end], length(original_x_coords)); crs=input_crs, dim=X),
        Y(LinRange(original_y_coords[1], original_y_coords[end], length(original_y_coords)); crs=input_crs, dim=Y)
    )

    filled_raster = Raster(final_data, new_dims; missingval=missingval(surface_raster))
    write(output_geotiff_path, filled_raster; force=true)

    @info "Processing finished. Output saved to: $(output_geotiff_path)"
    return output_geotiff_path
end


fill_missing_values

In [ ]:
# Get all files in data/raw
raw_dir = "data/raw"
files = filter(f -> endswith(f, ".tif"), readdir(raw_dir))

# Run interpolation on each file
for f in files
    try
        @info "Processing file: $f"
        output_path = fill_missing_values(f; lat_lon_len=10, epsilon2=10.0)
        @info "Successfully saved: $output_path"
    catch e
        @error "Failed processing $f" exception=(e, catch_backtrace())
    end
end


[ Info: Processing file: woa23_all_O00_01.nc_A_mn.tif
[ Info: Loading GeoTIFF from: data/raw/woa23_all_O00_01.nc_A_mn.tif
[ Info: Found 27059 valid points.


→ Reading subset: lon[1:172321] × lat[480:86400]
✓ Watermask applied. Kept 42810 of 64800 cells.


┌ Warning: Preconditioned conjugate gradients method did not converge
└ @ DIVAnd /opt/julia-depot/packages/DIVAnd/4UymR/src/DIVAnd_solve.jl:83


 23.828576 seconds (12.02 M allocations: 5.897 GiB, 14.55% gc time, 72.95% compilation time)


[ Info: Processing finished. Output saved to: data/filled/woa23_all_O00_01.nc_A_mn.tif.filled.2D.len10.e10.0.tif
[ Info: Successfully saved: data/filled/woa23_all_O00_01.nc_A_mn.tif.filled.2D.len10.e10.0.tif
[ Info: Processing file: woa23_all_O00_01.nc_A_sd.tif
[ Info: Loading GeoTIFF from: data/raw/woa23_all_O00_01.nc_A_sd.tif


...

┌ Warning: Preconditioned conjugate gradients method did not converge
└ @ DIVAnd /opt/julia-depot/packages/DIVAnd/4UymR/src/DIVAnd_solve.jl:83
[ Info: Processing finished. Output saved to: data/filled/woa23_all_o00_01_o_mn.tif.filled.2D.len10.e10.0.tif
[ Info: Successfully saved: data/filled/woa23_all_o00_01_o_mn.tif.filled.2D.len10.e10.0.tif
[ Info: Processing file: woa23_all_o00_01_o_sd.tif
[ Info: Loading GeoTIFF from: data/raw/woa23_all_o00_01_o_sd.tif


→ Reading subset: lon[1:172321] × lat[480:86400]


[ Info: Found 29485 valid points.


✓ Watermask applied. Kept 42810 of 64800 cells.
  7.273241 seconds (39.23 k allocations: 5.194 GiB, 31.58% gc time)


┌ Warning: Preconditioned conjugate gradients method did not converge
└ @ DIVAnd /opt/julia-depot/packages/DIVAnd/4UymR/src/DIVAnd_solve.jl:83
[ Info: Processing finished. Output saved to: data/filled/woa23_all_o00_01_o_sd.tif.filled.2D.len10.e10.0.tif
[ Info: Successfully saved: data/filled/woa23_all_o00_01_o_sd.tif.filled.2D.len10.e10.0.tif
[ Info: Processing file: woa23_all_p00_01.nc_p_mn.tif
[ Info: Loading GeoTIFF from: data/raw/woa23_all_p00_01.nc_p_mn.tif
